In [1]:
import os
import re
import spacy
import contractions
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings

C:\Users\HP\AppData\Local\Temp\ipykernel_17688\705940600.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:
loader = PyPDFLoader('file.pdf')
text = loader.load()
pages = []
for page in text:
    pages.append(page.page_content)
pages
data = ""
for i in pages:
    data += i
data

'==================================================================== \nARTIFICIAL INTELLIGENCE (AI), MACHINE LEARNING (ML) AND \nDEEP LEARNING (DL) - A COMPLETE GUIDE \n==================================================================== \n \nTable of Contents \n------------------ \n1. Introduction \n2. What is Artificial Intelligence (AI)? \n3. History and Evolution of AI \n4. Types of Artificial Intelligence \n5. What is Machine Learning (ML)? \n6. How Machine Learning Works \n7. Types of Machine Learning \n   7.1 Supervised Learning \n   7.2 Unsupervised Learning \n   7.3 Semi-Supervised Learning \n   7.4 Reinforcement Learning \n8. Common Machine Learning Algorithms \n9. What is Deep Learning (DL)? \n10. How Deep Learning Works \n11. Neural Network Architectures \n12. Difference Between AI, ML and DL \n13. Applications of AI, ML and DL \n14. Popular Tools and Frameworks \n15. Challenges and Limitations \n16. Ethics and Responsible AI \n17. Future Trends \n18. Conclusion \n19. Glos

In [3]:
data = data.lower()

In [4]:
data = contractions.fix(data)

In [5]:
data = re.sub('[^0-9a-zA-Z\s]', "", data).strip()
data = re.sub('[0-9]', "", data)
data = re.sub('\s+', " ", data).strip()
data

'artificial intelligence ai machine learning ml and deep learning dl a complete guide table of contents introduction what is artificial intelligence ai history and evolution of ai types of artificial intelligence what is machine learning ml how machine learning works types of machine learning supervised learning unsupervised learning semisupervised learning reinforcement learning common machine learning algorithms what is deep learning dl how deep learning works neural network architectures difference between ai ml and dl applications of ai ml and dl popular tools and frameworks challenges and limitations ethics and responsible ai future trends conclusion glossary of terms introduction artificial intelligence machine learning and deep learning are three of the most talkedabout technologies of the twentyfirst century they are reshaping industries changing how businesses operate and influencing everyday life in ways most people do not even notice from voice assistants like siri and alexa

In [6]:
nlp = spacy.load('en_core_web_sm')

In [7]:
tokens = nlp(data)
lemmatized_tokens = [token.lemma_ for token in tokens if not token.is_stop]
data = " ".join(lemmatized_tokens).strip()
data

'artificial intelligence ai machine learn ml deep learning dl complete guide table content introduction artificial intelligence ai history evolution ai type artificial intelligence machine learn ml machine learn work type machine learn supervise learning unsupervised learning semisupervise learn reinforcement learn common machine learning algorithm deep learning dl deep learning work neural network architecture difference ai ml dl application ai ml dl popular tool framework challenge limitation ethic responsible ai future trend conclusion glossary term introduction artificial intelligence machine learning deep learning talkedabout technology twentyfirst century reshape industry change business operate influence everyday life way people notice voice assistant like siri alexa recommendation system netflix amazon selfdrive car medical diagnosis tool technology term interchangeably casual conversation thing artificial intelligence broad concept machine learning subset ai deep learning subs

In [8]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 100
)
data = text_splitter.create_documents([data])
data

[Document(metadata={}, page_content='artificial intelligence ai machine learn ml deep learning dl complete guide table content introduction artificial intelligence ai history evolution ai type artificial intelligence machine learn ml machine learn work type machine learn supervise learning unsupervised learning semisupervise learn reinforcement learn common machine learning algorithm deep learning dl deep learning work neural network architecture difference ai ml dl application ai ml dl popular tool framework challenge limitation'),
 Document(metadata={}, page_content='architecture difference ai ml dl application ai ml dl popular tool framework challenge limitation ethic responsible ai future trend conclusion glossary term introduction artificial intelligence machine learning deep learning talkedabout technology twentyfirst century reshape industry change business operate influence everyday life way people notice voice assistant like siri alexa recommendation system netflix amazon self

In [9]:
embedding_model = HuggingFaceEmbeddings(
    model_name = 'sentence-transformers/all-miniLM-L6-V2'
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [10]:
vectordb = FAISS.from_documents(documents=data, embedding=embedding_model)
vectordb

In [11]:
verification_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a document verifier."
    "Your task is to verify whether the retrieved document is relevant to the user's question."
    "check the retrieved chunk score in percentage,"

    "example, question - what is machine learning, answer - Deep learning is the subset of ai."
    "wrong answer"
    "retrived out of 10 chunks 8 chunks are correct. return the score: "
    "if Score is 80"
    "give only 80"
    "don't give output like 80%"),
    
    ("human","User question: {query}"
      "Retrieved document:{r_chunks} "
    )
])

In [12]:
llm_model = ChatGoogleGenerativeAI(
    model = 'gemini-3.5-flash-lite',
    api_key = os.environ['GEMINI_API_KEY']
)

In [13]:
parser = StrOutputParser()

In [14]:
verification_chain = verification_prompt | llm_model | parser

In [15]:
generation_prompt = ChatPromptTemplate.from_messages([
    ('system', "you are a helpful assistant"
     'generate the output for the given user_prompt'),
    ("human","User question: {query}")
])

In [16]:
generation_chain = generation_prompt | llm_model | parser

In [17]:
def doc_analyzer(query, k=10):
    r_chunks = vectordb.similarity_search(query, k=k)
    r_chunks = [doc.page_content for doc in r_chunks]

    verification_response = verification_chain.invoke({'query' : query, 'r_chunks' : r_chunks})
    print(f"correct chunks : {verification_response}%")
    if int(verification_response) >= 80:
        response = generation_chain.invoke({'query' : query})
        return response
    else:
        return f"only {verification_response}% response are correct"

user_prompt = 'explain python'

response = doc_analyzer(user_prompt)
print(response)

correct chunks : 70%
only 70% response are correct
